In [1]:
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import json
from typing import Dict
from pathlib import Path


my_api_key = "API_key"

client = genai.Client(api_key = my_api_key)
model1 = "gemini-2.5-flash"
model2 = "gemini-3-pro-preview"
csv_path  = "Quine_LLM_tekstualni_manual.csv"

Koristio sam Googleove modele, maknuo sam API key jer se ne smije dijeliti

In [2]:
def load_manual(filepath: str) -> str:

    return pd.read_csv(filepath, sep = ";", quotechar = '"').set_index("index").to_string()

Funkcija za učitavanje manuala

In [3]:
def radical_translation(prompt: str, 
    model: str, 
    temperature: float, 
    n_iterations: int,
    manual: str
) -> Dict[str, str]:

    all_translations = {}
        
    formatted_prompt = prompt.format(manual=manual)

    config = types.GenerateContentConfig(
        temperature=temperature,
        system_instruction="Respond only with the proposed solutions for the translations of the words into English."
    )
    for i in range(n_iterations):
        response = client.models.generate_content(
        model = model,
        contents = formatted_prompt,
        config = config)
            
        all_translations[f"response_{i+1}"] = response.text
    
    return all_translations

Funkcija radikalnog prevođenja, stavio sam nekoliko varijabli da se mogu mijenjati za različite rezultate

In [4]:
def save_translations(translations: Dict[str, str], filename: str):

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(translations, f, indent=2, ensure_ascii=False)

Za spremanje odgovora, da se mogu analizirati

In [5]:
def main():

    manual = load_manual(csv_path)

    QUINE_PROMPT = """You are a linguist encountering an unknown language with no prior cultural knowledge.
                Your task: Translate each word/phrase from the manual based solely on the observational descriptions provided in the manual.
                
                This is the manual: {manual}
                
                Remember: You cannot assume any cultural concepts or patterns of structuring reality."""

    translations = radical_translation(QUINE_PROMPT, model1, 0.5, 15, manual)

    for key, text in translations.items():
        print(f"\n{key.replace('_', ' ').title()}:")
        print(text)
        print("-" * 80)


    save_translations(translations, "translations_output.json")
    print("\nTranslations saved to translations_output.json")

if __name__ == "__main__":
    main()


Response 1:
Blirox: a metal box with four wheels and doors that have glass windows
Varnon: a rectangular object with a hard cover whose insides are made out of pieces of paper attached together
Blirox julpa: the doors of the metal box with four wheels move so that the insides of the metal box are visible
Blirox flim: a metal box with four wheels and doors that have glass windows starts moving producing a noise
Varnon julpa: a rectangular object with a hard cover whose insides are made out of pieces of paper attached together is cracked open doubling its length
Blirox Blirox: two metal boxes with four wheels and doors that have glass windows
--------------------------------------------------------------------------------

Response 2:
Blirox: a metal box with four wheels and doors that have glass windows
Varnon: a rectangular object with a hard cover whose insides are made out of pieces of paper attached together
Blirox julpa: the doors of the metal box with four wheels move so that the

Glavna funkcija, probao sam različite Quine promptove, sadržaj promptova dosta utječe na odgovore, nekad model samo vrati cijele opise iz manuala kao prijevode,
a nekad daje jednu riječ ili frazu, svaki put daje točne odgovore s blagim varijacijama

In [6]:
def main():

    manual = load_manual(csv_path)

    QUINE_PROMPT_2 = """You are a linguist encountering an unknown language spoken by a remote tribe. You do not know anything about their culture or their language.
                    Without presupposing any structural features of their language, try to find the closest translations for the words/phrases in the manual.
                    This is the manual: {manual}"""

    translations = radical_translation(QUINE_PROMPT_2, model2, 0.7, 15, manual)

    for key, text in translations.items():
        print(f"\n{key.replace('_', ' ').title()}:")
        print(text)
        print("-" * 80)


    save_translations(translations, "translations_output_2.json")
    print("\nTranslations saved to translations_output_2.json")

if __name__ == "__main__":
    main()


Response 1:
Blirox: Car
Varnon: Book
julpa: Open / Opens
flim: Move / Moves
Blirox Blirox: Cars / Two cars
--------------------------------------------------------------------------------

Response 2:
Blirox: Car
Varnon: Book
julpa: Open
flim: Move
Blirox Blirox: Two cars
--------------------------------------------------------------------------------

Response 3:
**Blirox**: Car
**Varnon**: Book
**julpa**: Open (or "to open")
**flim**: Move (or "to move")
**Blirox Blirox**: Two cars (or "cars")
--------------------------------------------------------------------------------

Response 4:
Blirox: Car
Varnon: Book
Julpa: Open / Opens
Flim: Move / Start
Blirox Blirox: Two cars / Cars
--------------------------------------------------------------------------------

Response 5:
Blirox: Car
Varnon: Book
Julpa: Opens / Open
Flim: Moves / Starts
--------------------------------------------------------------------------------

Response 6:
Blirox: Car
Varnon: Book
julpa: Opens / To open
flim: M

In [7]:
def main():

    manual = load_manual(csv_path)

    QUINE_PROMPT_3 = """You are a linguist encountering an unknown language spoken by a remote tribe. You do not know anything about their culture or their language.
                    Specifically, remember not to assume any ontological constraints that implicitly structure reality for English speakers.                     
                    This is the manual: {manual}"""

    translations = radical_translation(QUINE_PROMPT_3, model2, 0.7, 15, manual)

    for key, text in translations.items():
        print(f"\n{key.replace('_', ' ').title()}:")
        print(text)
        print("-" * 80)


    save_translations(translations, "translations_output_3.json")
    print("\nTranslations saved to translations_output_3.json")

if __name__ == "__main__":
    main()


Response 1:
**Blirox:** Car (or Automobile)
**Varnon:** Book
**Julpa:** Open (specifically: the state of having the interior exposed or unfolded)
**Flim:** Move (specifically: active, noisy locomotion)
**[Reduplication of a noun]:** Two (or a pair of)
--------------------------------------------------------------------------------

Response 2:
**Blirox**: Car (or Automobile)
**Varnon**: Book
**julpa**: Open (or To open / Exposed insides)
**flim**: Move (or To move / In motion)
--------------------------------------------------------------------------------

Response 3:
**Blirox**: Car
**Varnon**: Book
**julpa**: Open / Opening / To reveal the inside
**flim**: Move / Moving / To run / To function
--------------------------------------------------------------------------------

Response 4:
**Blirox**: Car / Automobile
**Varnon**: Book
**Julpa**: Open / To open / To expose the interior
**Flim**: To move / To run / To activate
**[Repetition of a noun]**: Two / Pair of
--------------------